<a href="https://colab.research.google.com/github/DanidDACC/ACTIVIDADES-GUIADAS-ALGOTIMOS-/blob/main/AG3_Algoritmos(Colonia_de_Hormigas).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ACTIVIDAD GUIADA No. 3

#Algoritmos - Actividad Guiada 3

Nombre: Danid Camargo
URL: https://colab.research.google.com/drive/1TTn9BFTJssgAR-jANn2nZ0z6QfbJQgaZ#scrollTo=zivafJxWmMwq


In [ ]:
#Modulo de llamadas http para descargar ficheros
!pip install requests

#Libreria del problema TSP: http://elib.zib.de/pub/mp-testdata/tsp/tsplib/tsplib.html
!pip install tsplib95

In [ ]:
import tsplib95
import random
from math import e
import urllib.request

In [ ]:
import urllib.request
import tsplib95

# 1. DATOS DEL PROBLEMA (Usando servidor espejo)
#file = "swiss42.tsp"
#url = "https://raw.githubusercontent.com/mastqe/tsplib/master/swiss42.tsp"

# Descargar el archivo directamente sin compresión
urllib.request.urlretrieve(url, file)

# Cargar el archivo con tsplib95
problem = tsplib95.load(file)

# 2. NODOS
Nodos = list(problem.get_nodes())

# 3. FUNCIONES DE DISTANCIA
def distancia(a, b, problem):
    return problem.get_weight(a, b)

def distancia_total(solucion, problem):
    distancia_tot = 0
    for i in range(len(solucion)-1):
        distancia_tot += distancia(solucion[i], solucion[i+1], problem)
    # solucion[-1] es una forma más limpia en Python de obtener el último elemento
    return distancia_tot + distancia(solucion[-1], solucion[0], problem)

In [ ]:
#DATOS DEL PROBLEMA
file = "swiss42.tsp" ; urllib.request.urlretrieve("http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/swiss42.tsp.gz", file + '.gz')
!gzip -d swiss42.tsp.gz     #Descomprimir el fichero de datos
problem = tsplib95.load(file)

#Nodos
Nodos = list(problem.get_nodes())

#Devuelve la distancia entre dos nodos
def distancia(a,b, problem):
  return problem.get_weight(a,b)

#Devuelve la distancia total de una trayectoria/solucion(lista de nodos)
def distancia_total(solucion, problem):
  distancia_total = 0
  for i in range(len(solucion)-1):
    distancia_total += distancia(solucion[i] ,solucion[i+1] ,  problem)
  return distancia_total + distancia(solucion[len(solucion)-1] ,solucion[0], problem)


##Algoritmo de colonia de hormigas

La función Add_Nodo selecciona al azar un nodo con probabilidad uniforme.
Para ser mas eficiente debería seleccionar el próximo nodo siguiendo la probabilidad correspondiente a la ecuación:

$p^k_{ij}(t) = \frac{[\tau_{ij}(t)]^\alpha[\nu_{ij}]^\beta}{\sum_{l\in J^k_i} [\tau_{il}(t)]^\alpha[\nu_{il}]^\beta}$, si $j \in J^k_i$

$p^k_{ij}(t) = 0$, si $j \notin J^k_i$

In [ ]:
def Add_Nodo(problem, H ,T ) :
  #Mejora:Establecer una funcion de probabilidad para
  # añadir un nuevo nodo dependiendo de los nodos mas cercanos y de las feromonas depositadas
  Nodos = list(problem.get_nodes())
  return random.choice(   list(set(range(1,len(Nodos))) - set(H) )  )


def Incrementa_Feromona(problem, T, H ) :
  #Incrementa segun la calidad de la solución. Añadir una cantidad inversamente proporcional a la distancia total
  for i in range(len(H)-1):
    T[H[i]][H[i+1]] += 1000/distancia_total(H, problem)
  return T

def Evaporar_Feromonas(T ):
  #Evapora 0.3 el valor de la feromona, sin que baje de 1
  #Mejora:Podemos elegir diferentes funciones de evaporación dependiendo de la cantidad actual y de la suma total de feromonas depositadas,...
  T = [[ max(T[i][j] - 0.3 , 1) for i in range(len(Nodos)) ] for j in range(len(Nodos))]
  return T

In [ ]:
def hormigas(problem, N) :
  #problem = datos del problema
  #N = Número de agentes(hormigas)

  #Nodos
  Nodos = list(problem.get_nodes())
  #Aristas
  Aristas = list(problem.get_edges())

  #Inicializa las aristas con una cantidad inicial de feromonas:1
  #Mejora: inicializar con valores diferentes dependiendo diferentes criterios
  T = [[ 1 for _ in range(len(Nodos)) ] for _ in range(len(Nodos))]

  #Se generan los agentes(hormigas) que serán estructuras de caminos desde 0
  Hormiga = [[0] for _ in range(N)]

  #Recorre cada agente construyendo la solución
  for h in range(N) :
    #Para cada agente se construye un camino
    for i in range(len(Nodos)-1) :

      #Elige el siguiente nodo
      Nuevo_Nodo = Add_Nodo(problem, Hormiga[h] ,T )
      Hormiga[h].append(Nuevo_Nodo)

    #Incrementa feromonas en esa arista
    T = Incrementa_Feromona(problem, T, Hormiga[h] )
    #print("Feromonas(1)", T)

    #Evapora Feromonas
    T = Evaporar_Feromonas(T)
    #print("Feromonas(2)", T)

    #Seleccionamos el mejor agente
  mejor_solucion = []
  mejor_distancia = 10e100
  for h in range(N) :
    distancia_actual = distancia_total(Hormiga[h], problem)
    if distancia_actual < mejor_distancia:
      mejor_solucion = Hormiga[h]
      mejor_distancia =distancia_actual


  print(mejor_solucion)
  print(mejor_distancia)


hormigas(problem, 1000)

[0, 6, 40, 38, 31, 15, 36, 34, 20, 29, 19, 7, 32, 1, 14, 35, 33, 24, 17, 13, 16, 5, 28, 27, 10, 41, 9, 2, 11, 18, 12, 4, 39, 30, 22, 26, 21, 3, 8, 23, 25, 37]
3917


In [ ]:
import requests
import tsplib95

#file = "swiss42.tsp"
# URL espejo en GitHub que es mucho más estable
#url = "https://raw.githubusercontent.com/mastqe/tsplib/master/swiss42.tsp"

print("Conectando con el servidor espejo...")
respuesta = requests.get(url, timeout=10) # Agregamos un límite de tiempo por seguridad

# Verificamos que la descarga fue exitosa antes de guardar
if respuesta.status_code == 200:
    with open(file, 'wb') as f:
        f.write(respuesta.content)
    print("Descarga completada exitosamente.")
else:
    print(f"Error al descargar: Código {respuesta.status_code}")

# Cargar el problema
problem = tsplib95.load(file)
Nodos = list(problem.get_nodes())
print(f"Total de nodos cargados listos para usar: {len(Nodos)}")

Conectando con el servidor espejo...
Descarga completada exitosamente.
Total de nodos cargados listos para usar: 42


In [ ]:
import requests
import tsplib95

# DATOS DEL PROBLEMA
file = "swiss42.tsp"
# Utilizamos un mirror estable en GitHub para evitar el bloqueo del servidor original
url = "https://raw.githubusercontent.com/mastqe/tsplib/master/swiss42.tsp"

print("Descargando el fichero de datos...")
respuesta = requests.get(url)
with open(file, 'wb') as f:
    f.write(respuesta.content)
print("Descarga completada exitosamente.")

# Cargar el problema
problem = tsplib95.load(file)

# Nodos
Nodos = list(problem.get_nodes())

# Devuelve la distancia entre dos nodos
def distancia(a, b, problem):
    return problem.get_weight(a, b)

# Devuelve la distancia total de una trayectoria/solucion (lista de nodos)
def distancia_total(solucion, problem):
    distancia_tot = 0
    for i in range(len(solucion)-1):
        distancia_tot += distancia(solucion[i], solucion[i+1], problem)
    return distancia_tot + distancia(solucion[-1], solucion[0], problem)

Descargando el fichero de datos...
Descarga completada exitosamente.


In [ ]:
import random

def Add_Nodo(problem, H, T):
    Nodos = list(problem.get_nodes())
    # Selecciona un nodo disponible que no esté en el historial H
    return random.choice(list(set(range(1, len(Nodos))) - set(H)))

def Incrementa_Feromona(problem, T, H):
    dist_total = distancia_total(H, problem)
    for i in range(len(H)-1):
        T[H[i]][H[i+1]] += 1000 / dist_total
    return T

def Evaporar_Feromonas(T, Nodos):
    # Evapora 0.3 el valor de la feromona, sin que baje de 1
    T = [[max(T[i][j] - 0.3, 1) for i in range(len(Nodos))] for j in range(len(Nodos))]
    return T

In [ ]:
def hormigas(problem, N):
    Nodos = list(problem.get_nodes())

    # Inicializa las aristas con una cantidad inicial de feromonas: 1
    T = [[1 for _ in range(len(Nodos))] for _ in range(len(Nodos))]

    # Se generan los agentes (hormigas) que serán estructuras de caminos desde el nodo 0
    Hormiga = [[0] for _ in range(N)]

    # Recorre cada agente construyendo la solución
    for h in range(N):
        # Para cada agente se construye un camino
        for i in range(len(Nodos)-1):
            Nuevo_Nodo = Add_Nodo(problem, Hormiga[h], T)
            Hormiga[h].append(Nuevo_Nodo)

        # Incrementa feromonas en esa arista
        T = Incrementa_Feromona(problem, T, Hormiga[h])

        # Evapora Feromonas pasando la lista de Nodos
        T = Evaporar_Feromonas(T, Nodos)

    # Seleccionamos el mejor agente
    mejor_solucion = []
    mejor_distancia = float('inf') # Uso de infinito estándar de Python

    for h in range(N):
        distancia_actual = distancia_total(Hormiga[h], problem)
        if distancia_actual < mejor_distancia:
            mejor_solucion = Hormiga[h]
            mejor_distancia = distancia_actual

    print("Mejor ruta encontrada:", mejor_solucion)
    print("Distancia mínima:", mejor_distancia)

# Ejecutar el algoritmo
hormigas(problem, 1000)

Mejor ruta encontrada: [0, 28, 15, 36, 17, 25, 22, 26, 6, 18, 11, 19, 1, 33, 38, 39, 16, 7, 5, 23, 20, 32, 27, 9, 29, 41, 40, 21, 8, 10, 24, 35, 37, 13, 3, 30, 12, 4, 2, 14, 34, 31]
Distancia mínima: 3855


In [ ]:
import requests
import tsplib95
import random

# 1. Descarga segura de los datos
file = "swiss42.tsp"
url = "https://raw.githubusercontent.com/mastqe/tsplib/master/swiss42.tsp"

print("Conectando con el servidor espejo...")
respuesta = requests.get(url, timeout=10)

if respuesta.status_code == 200:
    with open(file, 'wb') as f:
        f.write(respuesta.content)
    print("Descarga completada exitosamente.")
else:
    print(f"Error al descargar: Código {respuesta.status_code}")

# 2. Inicialización del problema
problem = tsplib95.load(file)
Nodos = list(problem.get_nodes())

def distancia(a, b, problem):
    return problem.get_weight(a, b)

def distancia_total(solucion, problem):
    distancia_tot = 0
    for i in range(len(solucion)-1):
        distancia_tot += distancia(solucion[i], solucion[i+1], problem)
    return distancia_tot + distancia(solucion[-1], solucion[0], problem)

# 3. Lógica de Colonia de Hormigas
def Add_Nodo(problem, H, T):
    Nodos = list(problem.get_nodes())
    return random.choice(list(set(range(1, len(Nodos))) - set(H)))

def Incrementa_Feromona(problem, T, H):
    dist_total = distancia_total(H, problem)
    for i in range(len(H)-1):
        T[H[i]][H[i+1]] += 1000 / dist_total
    return T

def Evaporar_Feromonas(T, Nodos):
    return [[max(T[i][j] - 0.3, 1) for i in range(len(Nodos))] for j in range(len(Nodos))]

def hormigas(problem, N):
    Nodos = list(problem.get_nodes())
    T = [[1 for _ in range(len(Nodos))] for _ in range(len(Nodos))]
    Hormiga = [[0] for _ in range(N)]

    for h in range(N):
        for i in range(len(Nodos)-1):
            Nuevo_Nodo = Add_Nodo(problem, Hormiga[h], T)
            Hormiga[h].append(Nuevo_Nodo)

        T = Incrementa_Feromona(problem, T, Hormiga[h])
        T = Evaporar_Feromonas(T, Nodos)

    mejor_solucion = []
    mejor_distancia = float('inf')

    for h in range(N):
        distancia_actual = distancia_total(Hormiga[h], problem)
        if distancia_actual < mejor_distancia:
            mejor_solucion = Hormiga[h]
            mejor_distancia = distancia_actual

    print("Mejor ruta encontrada:", mejor_solucion)
    print("Distancia mínima:", mejor_distancia)

# 4. Ejecución
hormigas(problem, 1000)

Conectando con el servidor espejo...
Descarga completada exitosamente.
Mejor ruta encontrada: [0, 30, 23, 22, 38, 25, 2, 6, 26, 34, 12, 10, 7, 19, 15, 17, 21, 39, 8, 32, 28, 31, 40, 29, 24, 9, 41, 3, 35, 37, 36, 27, 1, 13, 18, 20, 33, 4, 5, 16, 14, 11]
Distancia mínima: 3811
